# Qwen2-VL Baseline: Zero-Shot + LoRA Fine-Tuning

This notebook uses `Qwen/Qwen2-VL-2B-Instruct` as a stronger baseline. The first section runs zero-shot evaluation. The second section attaches LoRA adapters and fine-tunes only adapter weights.

## 1. Colab Repository Setup

Run this first in Colab. It clones or updates the repository and changes the working directory to the repo root.

In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/kdnehihi/multi-task-moe-vlm-assistant.git"
COLAB_REPO_DIR = Path("/content/multi-task-moe-vlm-assistant")

if Path("/content").exists():
    if not COLAB_REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(COLAB_REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "pull", "origin", "main"], check=True, cwd=COLAB_REPO_DIR)
    os.chdir(COLAB_REPO_DIR)

print(f"Current working directory: {Path.cwd()}")

## 2. Setup

In [ ]:
from pathlib import Path
import gc
import json
import os
import random
import subprocess
import sys

os.environ.setdefault("CUDA_LAUNCH_BLOCKING", "1")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

PROJECT_ROOT = Path.cwd()
sys.path.append(str(PROJECT_ROOT))

required_packages = ["qwen-vl-utils", "accelerate"]
for package_name in required_packages:
    import_name = package_name.replace("-", "_")
    try:
        __import__(import_name)
    except ModuleNotFoundError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package_name], check=True)

import torch

# Colab may preinstall an old torchao version. Recent PEFT checks torchao when it
# exists, even though this notebook does not use torchao quantization.
try:
    import torchao
    torchao_version = getattr(torchao, "__version__", "0.0.0")
    major, minor, *_ = [int(part) for part in torchao_version.split(".")[:2]]
    if (major, minor) < (0, 16):
        print(f"Removing incompatible torchao=={torchao_version} before importing PEFT.")
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=True)
        for module_name in list(sys.modules):
            if module_name == "torchao" or module_name.startswith("torchao."):
                del sys.modules[module_name]
except ModuleNotFoundError:
    pass

try:
    import peft
except ModuleNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft"], check=True)

from peft import LoraConfig, get_peft_model
from PIL import Image
from qwen_vl_utils import process_vision_info
from torch.utils.data import DataLoader, Dataset
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration

torch.__version__

## 3. Config

In [ ]:
METADATA_PATH = PROJECT_ROOT / "data/processed/multitask/validation.jsonl"
MODEL_NAME = "Qwen/Qwen2-VL-2B-Instruct"
LOCAL_CHECKPOINT_DIR = PROJECT_ROOT / "outputs/checkpoints/qwen2vl_lora_baseline_sample"
DRIVE_CHECKPOINT_DIR = Path("/content/drive/MyDrive/multi-task-moe-vlm-assistant/checkpoints/qwen2vl_lora_baseline_sample")
USE_GOOGLE_DRIVE_CHECKPOINT = True

MAX_SAMPLES = 50
ZERO_SHOT_LIMIT = 20
BATCH_SIZE = 1
EPOCHS = 1
LEARNING_RATE = 1e-4
GRADIENT_ACCUMULATION_STEPS = 4
SEED = 42
MAX_NEW_TOKENS = 32
MIN_PIXELS = 256 * 28 * 28
MAX_PIXELS = 512 * 28 * 28

LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "v_proj"]

DATA_LIMITS = {"docvqa": 1667, "chartqa": 1667, "textvqa": 1666}
EXPECTED_TOTAL_SAMPLES = sum(DATA_LIMITS.values())

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_MIXED_PRECISION = DEVICE == "cuda"
TORCH_DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
DEVICE

## 4. Checkpoint Storage

In [ ]:
CHECKPOINT_DIR = LOCAL_CHECKPOINT_DIR

if USE_GOOGLE_DRIVE_CHECKPOINT:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        CHECKPOINT_DIR = DRIVE_CHECKPOINT_DIR
    except Exception as error:
        print(f"Google Drive checkpoint storage is unavailable: {error}")
        CHECKPOINT_DIR = LOCAL_CHECKPOINT_DIR

print(f"Checkpoint directory: {CHECKPOINT_DIR}")

## 5. Prepare Data If Needed

In [ ]:
def count_jsonl_lines(path: Path) -> int:
    if not path.exists():
        return 0
    with path.open("r", encoding="utf-8") as f:
        return sum(1 for _ in f)


current_count = count_jsonl_lines(METADATA_PATH)
print(f"Current processed examples: {current_count}")

if current_count < EXPECTED_TOTAL_SAMPLES:
    for dataset_name, limit in DATA_LIMITS.items():
        command = [
            sys.executable,
            str(PROJECT_ROOT / "scripts/prepare_data.py"),
            "--dataset",
            dataset_name,
            "--split",
            "validation",
            "--limit",
            str(limit),
            "--streaming",
        ]
        print("Running:", " ".join(command))
        subprocess.run(command, check=True, cwd=PROJECT_ROOT)

    build_command = [sys.executable, str(PROJECT_ROOT / "scripts/build_multitask_data.py"), "--split", "validation"]
    print("Running:", " ".join(build_command))
    subprocess.run(build_command, check=True, cwd=PROJECT_ROOT)
else:
    print("Processed data already exists. Skipping data preparation.")

## 6. Load Records

In [ ]:
random.seed(SEED)
torch.manual_seed(SEED)

records = []
with METADATA_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))

random.shuffle(records)
records = records[:MAX_SAMPLES]

len(records), records[0]

# Part A: Zero-Shot Qwen2-VL Baseline

## 7. Load Zero-Shot Model

In [ ]:
def free_gpu_memory() -> None:
    for name in ["model", "optimizer", "scaler"]:
        if name in globals():
            del globals()[name]
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


free_gpu_memory()
processor = AutoProcessor.from_pretrained(
    MODEL_NAME,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS,
)
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=TORCH_DTYPE,
    device_map="auto" if DEVICE == "cuda" else None,
)
if DEVICE != "cuda":
    model.to(DEVICE)
model.eval()

## 8. Zero-Shot Prediction Helper

In [ ]:
def build_messages(image_path: str, question: str, answer: str | None = None):
    user_content = [
        {"type": "image", "image": str(PROJECT_ROOT / image_path)},
        {"type": "text", "text": f"Answer the visual question briefly. Question: {question}"},
    ]
    messages = [{"role": "user", "content": user_content}]
    if answer is not None:
        messages.append({"role": "assistant", "content": [{"type": "text", "text": answer}]})
    return messages


def qwen_predict(example: dict) -> str:
    messages = build_messages(example["image_path"], example["question"])
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )
    inputs = inputs.to(model.device)

    with torch.inference_mode():
        generated_ids = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS)

    generated_trimmed = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(inputs.input_ids, generated_ids)
    ]
    return processor.batch_decode(generated_trimmed, skip_special_tokens=True)[0].strip()

## 9. Run Zero-Shot Samples

In [ ]:
zero_shot_records = []
for index, example in enumerate(records[:ZERO_SHOT_LIMIT], start=1):
    prediction = qwen_predict(example)
    zero_shot_records.append({
        "question": example["question"],
        "answers": example["answers"],
        "prediction": prediction,
        "dataset": example["dataset"],
        "task_type": example["task_type"],
    })
    print(f"[{index}/{min(ZERO_SHOT_LIMIT, len(records))}] {example['dataset']} -> {prediction!r}")

zero_shot_records[:3]

# Part B: Qwen2-VL LoRA Fine-Tuning

## 10. Dataset and Collate Function

In [ ]:
class QwenVQAFinetuneDataset(Dataset):
    def __init__(self, examples):
        self.examples = examples

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        example = self.examples[index]
        answer = example["answers"][0] if example["answers"] else ""
        return {
            "image_path": example["image_path"],
            "question": example["question"],
            "answer": answer,
        }


def collate_fn(batch):
    full_messages = [build_messages(item["image_path"], item["question"], item["answer"]) for item in batch]
    prompt_messages = [build_messages(item["image_path"], item["question"]) for item in batch]

    full_texts = [processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=False) for messages in full_messages]
    prompt_texts = [processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True) for messages in prompt_messages]
    image_inputs, video_inputs = process_vision_info(full_messages)

    inputs = processor(
        text=full_texts,
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )
    prompt_inputs = processor(
        text=prompt_texts,
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )
    labels = inputs["input_ids"].clone()

    for row_index in range(labels.shape[0]):
        prompt_length = int(prompt_inputs["attention_mask"][row_index].sum().item())
        labels[row_index, :prompt_length] = -100

    pad_token_id = processor.tokenizer.pad_token_id
    if pad_token_id is not None:
        labels[labels == pad_token_id] = -100
    inputs["labels"] = labels
    return inputs


train_dataset = QwenVQAFinetuneDataset(records)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
len(train_dataset)

## 11. Inspect One Batch

In [ ]:
debug_batch = next(iter(train_loader))
for key, value in debug_batch.items():
    print(key, value.shape, value.dtype)

valid_labels = debug_batch["labels"][debug_batch["labels"] != -100]
print("label min/max:", int(valid_labels.min()), int(valid_labels.max()))
print("tokenizer vocab size:", processor.tokenizer.vocab_size)
assert int(valid_labels.min()) >= 0
assert int(valid_labels.max()) < processor.tokenizer.vocab_size
print("Batch check passed.")

## 12. Attach LoRA Adapter

In [ ]:
free_gpu_memory()
processor = AutoProcessor.from_pretrained(
    MODEL_NAME,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS,
)
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=TORCH_DTYPE,
    device_map="auto" if DEVICE == "cuda" else None,
)
model.config.use_cache = False

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGET_MODULES,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
if DEVICE != "cuda":
    model.to(DEVICE)
model.train()
model.print_trainable_parameters()

optimizer = torch.optim.AdamW([parameter for parameter in model.parameters() if parameter.requires_grad], lr=LEARNING_RATE)

## 13. Train LoRA

In [ ]:
if DEVICE == "cuda":
    torch.cuda.empty_cache()

autocast_device = "cuda" if DEVICE == "cuda" else "cpu"
scaler = torch.cuda.amp.GradScaler(enabled=USE_MIXED_PRECISION)
loss_history = []
optimizer.zero_grad(set_to_none=True)

for epoch in range(EPOCHS):
    for step, batch in enumerate(train_loader, start=1):
        batch = {key: value.to(model.device) if hasattr(value, "to") else value for key, value in batch.items()}

        with torch.autocast(device_type=autocast_device, dtype=torch.float16, enabled=USE_MIXED_PRECISION):
            outputs = model(**batch)
            loss = outputs.loss

        scaler.scale(loss / GRADIENT_ACCUMULATION_STEPS).backward()

        if step % GRADIENT_ACCUMULATION_STEPS == 0 or step == len(train_loader):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        loss_value = float(loss.detach().cpu())
        loss_history.append(loss_value)

        if step % 10 == 0 or step == 1:
            print(f"epoch={epoch + 1} step={step} loss={loss_value:.4f}")

len(loss_history), loss_history[:5], loss_history[-5:]

## 14. Save LoRA Adapter Checkpoint

In [ ]:
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(CHECKPOINT_DIR)
processor.save_pretrained(CHECKPOINT_DIR)
print(f"Saved Qwen2-VL LoRA checkpoint to {CHECKPOINT_DIR}")

## 15. Quick Fine-Tuned Inference Check

In [ ]:
model.eval()
prediction = qwen_predict(records[0])
print("question:", records[0]["question"])
print("prediction:", prediction)
print("answers:", records[0]["answers"])